# Production hardening for Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

What to get right before a Bedrock workload carries real traffic: credentials, retries
and timeouts, data retention, client-side metrics, output handling, a Region check at
start-up, and Guardrails, measured on every API that accepts them. Projects and cost
attribution are in
[`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb), and
quotas and service tiers in
[`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb).

**Covered:** a thread-safe key provider, retries with backoff and timeouts, retention
settings, client-side metrics, defensive output handling, Region pre-flight checks on
both endpoints, Guardrails on each API, and a pre-launch checklist.

## Setup

The notebook calls `bedrock-mantle` over HTTPS and `bedrock-runtime` through boto3 and
the SDKs. `post` from `_shared/bedrock.py`, this repository's helper module, makes the
control-plane calls and returns `(status, body)` instead of raising, and `err` reads an
error body with identifiers redacted, because this output is committed to a public
repository. The Guardrails section creates one throwaway guardrail and deletes it at the
end.

In [1]:
import concurrent.futures as cf
import email.message  # a stand-in for HTTPError.headers in section 2
import email.utils  # Retry-After in its HTTP-date form
import json
import math  # isfinite, to reject a nan or inf Retry-After
import random
import sys
import threading
import time
import urllib.error
import urllib.request
import uuid
from datetime import datetime, timedelta, timezone

from aws_bedrock_token_generator import provide_token

sys.path.insert(0, "../_shared")
from bedrock import (api_prefix, err, parse_json_lenient, post, resolve_runtime_id,
                     response_text)

REGION = "us-east-1"
MODEL = "google.gemma-4-31b"  # on bedrock-mantle; section 6 checks it, among models chosen to fail
PREFIX = "/openai/v1"

## 1. Credentials

Short-term Bedrock API keys are presigned SigV4 requests, so they carry the minting
role's permissions, last at most 12 hours, cannot be refreshed, and work only in the
Region they were minted for. Mint them in the process from the role it runs as, keep the
lifetime short, and never ship a long-term key, which is a static IAM user credential; a
service control policy can block long-term keys across an organisation, as the IAM
section of
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb)
shows. The provider below gives many threads one key and replaces it before it expires.
SigV4 signing needs no key at all.

In [2]:
class TokenProvider:
    """Short-term Bedrock API keys, shared across threads and replaced before they expire."""

    def __init__(self, region: str = REGION, ttl: timedelta = timedelta(minutes=15)):
        self.region, self.ttl, self.mints = region, ttl, 0
        self._token, self._renew_at = None, datetime.min.replace(tzinfo=timezone.utc)
        self._lock = threading.Lock()

    def get(self) -> str:
        with self._lock:  # one mint, however many threads ask at once
            now = datetime.now(timezone.utc)
            if now >= self._renew_at:
                self._token = provide_token(region=self.region, expiry=self.ttl)
                self._renew_at = now + self.ttl - timedelta(minutes=2)
                self.mints += 1
            return self._token


tokens = TokenProvider()
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    keys = list(pool.map(lambda _: tokens.get(), range(8)))
print(f"8 concurrent callers -> {tokens.mints} mint, all given the same key: {len(set(keys)) == 1}")

8 concurrent callers -> 1 mint, all given the same key: True


## 2. Retries and timeouts

Throttles (429), server errors (5xx) and the Messages API's `overloaded` (529) are
ordinary under load, because most models share capacity with no published quota. Retry
them with exponential backoff and jitter, and when the response carries a
`Retry-After` header use the wait it names instead of your own schedule. The helper below
caps that wait at 60 seconds rather than dropping a longer one: the backoff tops out
below that ceiling, so dropping a `Retry-After: 600` would come back about 8 seconds in,
sooner than the service asked. At the ceiling the caller still returns before the service
asked, by a factor of ten on that header, and the five-attempt loop below sleeps four
times, so a persistent `Retry-After: 600` holds one call for about four minutes. A caller
with a deadline should bound the total wait rather than the single one, and a caller that
can afford to wait should honour the full value, or hand the request to a queue and fail
it, rather than return early. The OpenAI and Anthropic SDKs drop a header above the same
ceiling and fall back to their own backoff, as does `post` in `_shared/bedrock.py`
(`0 < retry_after <= 60` in both SDKs at openai 2.50.0 and anthropic 0.120.2, and
`0 < after <= _RETRY_AFTER_MAX` in `post`, read 4 November 2026), so code that leans on
SDK retries alone comes back early under a long shed.

A 400 is a fault in the request and should
fail at once, with one exception: `bedrock-mantle` can return a server fault with a 400
status and `Internal server error` in the body, so the policy reads the body as well as
the status. Every call has a client timeout, because a wrong path can stall rather than
fail, and a retry loop without a timeout turns one stall into a long hang. On the boto3
path both the timeout and the retry policy belong to the client rather than the call: a
bare `boto3.client("bedrock-runtime")` reports `read_timeout` 60, `connect_timeout` 60
and `retries {'mode': 'legacy'}` (boto3 1.43.59, 4 November 2026), so pass a
`botocore.config.Config` carrying both to every Bedrock client, as section 7 does.

In [3]:
TRANSIENT = {429, 500, 502, 503, 504, 529}
SERVER_FAULT = ("internal server error", "internal failure", "internal error")
RETRY_AFTER_MAX = 60.0  # the ceiling this notebook caps a Retry-After at; see retry_delay


def should_retry(status: int, body: dict) -> bool:
    """Retry throttles and server faults, including a 4xx whose body reports a server fault."""
    if status in TRANSIENT:
        return True
    if 400 <= status < 500:
        return any(marker in json.dumps(body, default=str).lower() for marker in SERVER_FAULT)
    return False


def retry_delay(attempt: int, headers=None) -> float:
    """Seconds to wait: Retry-After when the service names one, else exponential backoff.

    Reads both forms HTTP allows, delta-seconds and an HTTP date. A value above
    RETRY_AFTER_MAX is capped there rather than dropped, so the wait stays as close to
    what the service asked as the ceiling allows instead of collapsing to a shorter
    backoff. At the ceiling the caller still returns before the service asked; section 2
    says what a caller that can afford to wait should do instead. Jitter goes on top
    either way, so clients shed together do not all return at the same moment.
    """
    raw = headers.get("Retry-After") if headers is not None else None
    after = None
    if raw is not None:
        try:
            after = float(raw)
        except ValueError:
            parsed = email.utils.parsedate_tz(raw)  # None, never raises, on anything else
            try:  # mktime_tz does raise, on a date it reads and cannot represent
                after = email.utils.mktime_tz(parsed) - time.time() if parsed else None
            except (ValueError, OverflowError):
                after = None
    # float() reads "nan" and "inf" and neither is a wait: nan fails every comparison,
    # and inf would cap to the ceiling as though the service had asked for it. Drop both
    # and let the backoff decide, as `_shared/bedrock.py` does.
    named = after is not None and math.isfinite(after) and after > 0
    # Capped, not dropped. The backoff tops out below the ceiling, so dropping a long
    # header would retry sooner than the service asked: a Retry-After of 600 would come
    # back about 8s in. The SDKs and `post` in `_shared/bedrock.py` do drop it (section 2).
    wait = min(after, RETRY_AFTER_MAX) if named else min(2 ** attempt, 16)
    return wait + random.random()  # nosec B311 - jitter, not security


def read_json(raw: bytes) -> dict:
    """A response body as a dict, whatever came back."""
    try:
        parsed = json.loads(raw or b"{}")
    except ValueError:
        return {"raw": raw[:500].decode("utf-8", "replace")}
    return parsed if isinstance(parsed, dict) else {"raw": parsed}

In [4]:
def resilient_call(path: str, body: dict, *, attempts: int = 5, timeout: int = 60) -> tuple[int, dict, int]:
    """POST to bedrock-mantle and retry what should_retry allows. Returns (status, body, attempts used)."""
    url = f"https://bedrock-mantle.{REGION}.api.aws{path}"
    if not url.startswith("https://"):
        raise ValueError("HTTPS only")
    for attempt in range(1, attempts + 1):
        headers = None  # set on every path that reaches the sleep below
        request = urllib.request.Request(url, data=json.dumps(body).encode(), method="POST", headers={
            "Authorization": f"Bearer {tokens.get()}", "Content-Type": "application/json"})
        try:
            # nosemgrep: dynamic-urllib-use-detected - the scheme is checked above
            with urllib.request.urlopen(request, timeout=timeout) as response:  # nosec B310  # noqa: S310
                return response.status, read_json(response.read()), attempt
        except urllib.error.HTTPError as exc:
            status, parsed = exc.code, read_json(exc.read())
            if not should_retry(status, parsed) or attempt == attempts:
                return status, parsed, attempt
            headers = exc.headers  # Retry-After, when the service sends it
        except OSError:  # a timeout or a dropped connection
            if attempt == attempts:
                return -1, {"error": {"message": "no response within the timeout"}}, attempt
        time.sleep(retry_delay(attempt, headers))
    raise RuntimeError("unreachable")

In [5]:
status, body, used = resilient_call(f"{PREFIX}/responses",
                                    {"model": MODEL, "input": "Reply with exactly: OK", "max_output_tokens": 64, "store": False})
print(f"a healthy call        -> HTTP {status} after {used} attempt, status {body.get('status')}: "
      f"{response_text(body).strip()!r}")

started = time.perf_counter()
status, body, used = resilient_call(f"{PREFIX}/responses", {"model": MODEL, "input": "Hi", "max_output_tokens": 8, "store": False})
print(f"max_output_tokens=8   -> HTTP {status} after {used} attempt, {time.perf_counter() - started:.1f}s: {err(body, 80)}")

print()
for label, (code_, sample) in {
    "a 400 reporting a server fault": (400, {"error": {"message": "Internal server error"}}),
    "a 400 about a parameter": (400, {"error": {"message": "Invalid 'max_output_tokens'"}}),
    "429": (429, {}),
    "529": (529, {}),
}.items():
    print(f"should_retry({label}) -> {should_retry(code_, sample)}")

# The wait, with the backoff for comparison.
def sent(value: str) -> email.message.Message:
    """A stand-in for HTTPError.headers: the same type, and case-insensitive too."""
    held = email.message.Message()
    held["retry-after"] = value  # lowercase on purpose; the lookup ignores case
    return held


for label, headers in {"no Retry-After": None, "Retry-After: 7": sent("7"),
                       "an HTTP date 12s out": sent(email.utils.formatdate(time.time() + 12, usegmt=True)),
                       "Retry-After: 600, capped at 60": sent("600"),
                       "Retry-After: nonsense": sent("soon"),
                       "a date out of range": sent("Tue, 20 Oct 99999 00:00:00 GMT")}.items():
    print(f"retry_delay(attempt=3, {label:34}) -> {retry_delay(3, headers):5.2f}s")

a healthy call        -> HTTP 200 after 1 attempt, status completed: 'OK'
max_output_tokens=8   -> HTTP 400 after 1 attempt, 0.0s: Invalid 'max_output_tokens': integer below minimum value. Expected a value >= 16

should_retry(a 400 reporting a server fault) -> True
should_retry(a 400 about a parameter) -> False
should_retry(429) -> True
should_retry(529) -> True
retry_delay(attempt=3, no Retry-After                    ) ->  8.75s
retry_delay(attempt=3, Retry-After: 7                    ) ->  7.83s
retry_delay(attempt=3, an HTTP date 12s out              ) -> 12.54s
retry_delay(attempt=3, Retry-After: 600, capped at 60    ) -> 60.50s
retry_delay(attempt=3, Retry-After: nonsense             ) ->  8.71s
retry_delay(attempt=3, a date out of range               ) ->  8.19s


In [6]:
# Grok 4.3 on the bare /v1 path, which is not its path: the service does not answer.
started = time.perf_counter()
status, body, used = resilient_call("/v1/responses", {"model": "xai.grok-4.3", "input": "Hi", "max_output_tokens": 16, "store": False},
                                    attempts=1, timeout=20)
elapsed = time.perf_counter() - started
print(f"Grok 4.3 on /v1/responses -> {status} after {elapsed:.0f}s: {err(body, 60)}")
print("the client timeout ended the call" if status == -1 else "the service answered")

Grok 4.3 on /v1/responses -> -1 after 20s: no response within the timeout
the client timeout ended the call


## 3. Data retention

Two settings decide what is kept. `store` on the Responses API defaults to `true` and
keeps the input and output for 30 days, so set it explicitly on every request. The data
retention mode applies to the account, a project or a model
([`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)
explains the scopes), and a model that does not allow the effective mode refuses calls:
its `GET /v1/models/{model}` reports `status` `unavailable` and says why. Read that
before you debug a request that fails for no visible reason.

In [7]:
code_, account = post("/v1/data_retention", None, region=REGION, method="GET")
print("account retention mode:", account.get("mode"))

for store in (None, False):
    request = {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16} | ({} if store is None else {"store": store})
    status, reply, _ = resilient_call(f"{PREFIX}/responses", request)
    print(f"store {'omitted' if store is None else store!s:7} -> the reply says store={reply.get('store')}")
    if reply.get("store"):
        # Do not keep it, and read the status: post() reports an HTTP error rather than raising.
        gone, confirmation = post(f"{PREFIX}/responses/{reply['id']}", None, region=REGION, method="DELETE")
        print(f"   deleted the stored reply: HTTP {gone}, deleted={confirmation.get('deleted')}")

for model in (MODEL, "anthropic.claude-fable-5"):
    code_, info = post(f"/v1/models/{model}", None, region=REGION, method="GET")
    retention = info.get("data_retention") or {}
    print(f"{model:26} status={info.get('status')!s:12} allowed modes={sorted(retention.get('allowed_modes', []))}")
    if info.get("status_reason"):
        print("   reason:", info["status_reason"])

account retention mode: inherit


store omitted -> the reply says store=True
   deleted the stored reply: HTTP 200, deleted=True


store False   -> the reply says store=False
google.gemma-4-31b         status=available    allowed modes=['aws_review', 'default', 'none', 'provider_data_share']
anthropic.claude-fable-5   status=unavailable  allowed modes=['aws_review', 'provider_data_share']
   reason: This model is not available under data retention mode 'default'.


## 4. Client-side metrics

The `AWS/BedrockMantle` CloudWatch namespace leaves gaps that matter under load. It
publishes no server-error metric and no latency metric, and its one error metric,
`InferenceClientErrors`, counts only what the service accepted for processing and then
failed, which leaves out whatever was rejected before that, quota-related 429s included.
There is no `InvocationThrottles` equivalent either, so by that definition throttling
does not show in the namespace at all, and the [mantle metrics
page](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring-mantle-metrics.html),
read on 19 October 2026, says to track HTTP 429 responses in the application instead.
`ListMetrics` on both namespaces in `us-east-1` on 19 October 2026 returned 8 metric
names under `AWS/BedrockMantle`, with no throttle, server-error or latency metric among
them, against 14 under `AWS/Bedrock` that include `InvocationThrottles`,
`InvocationServerErrors` and `InvocationLatency`; a listing reports what an account has
published, so read it as corroboration of the page rather than proof on its own. Section
6 of [`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)
measures the same gap.

Count outcomes and latency in the client as well. The recorder below sorts each call into
ok, throttled, server error, client error or timeout, keeping throttles in their own
bucket for the reason above, counts the retries, and reports p50 and p95 latency by
nearest rank. A call that needed a retry shows in the latency as well as in the count.

In [8]:
class CallMetrics:
    """Outcome counts and latency percentiles, recorded by the client."""

    def __init__(self):
        self.counts = dict.fromkeys(("ok", "throttled", "server_error", "client_error", "timeout"), 0)
        self.latencies: list[float] = []
        self.retries = 0

    def record(self, status: int, seconds: float, attempts: int = 1) -> None:
        self.latencies.append(seconds)
        self.retries += attempts - 1
        kind = ("ok" if status == 200 else "throttled" if status == 429 else "timeout" if status == -1
                else "server_error" if status >= 500 else "client_error")
        self.counts[kind] += 1

    def percentile(self, p: int) -> float:
        ordered = sorted(self.latencies)
        return ordered[max(0, -(-len(ordered) * p // 100) - 1)]  # nearest rank


metrics = CallMetrics()
for i in range(6):
    started = time.perf_counter()
    status, _, attempts = resilient_call(f"{PREFIX}/responses", {"model": MODEL, "input": f"Reply OK ({i})",
                                                                 "max_output_tokens": 16, "store": False})
    metrics.record(status, time.perf_counter() - started, attempts)
print(metrics.counts, f"| retries={metrics.retries} | p50={metrics.percentile(50):.2f}s "
      f"p95={metrics.percentile(95):.2f}s n={len(metrics.latencies)}")

{'ok': 6, 'throttled': 0, 'server_error': 0, 'client_error': 0, 'timeout': 0} | retries=0 | p50=0.25s p95=1.94s n=6


## 5. Defensive output handling

HTTP 200 is not yet a usable result. A reply can stop at the token limit (`status`
`incomplete` on the Responses API, `finish_reason` `length` on Chat Completions), a JSON
object can arrive with text after it, a model can answer in prose when a tool was
forced, and valid JSON can hold a wrong value. The function below checks `status` and
raises the budget when a reply was cut off, parses with `parse_json_lenient`, which
tolerates text after the object, and checks the required keys; the cell then compares
the values with the source. It starts from a deliberately small budget, so the first
attempt is cut off.

In [9]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}
PROMPT = ("Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the total as a "
          f"number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}")


def structured(model: str, prompt: str, schema: dict, budget: int = 24, attempts: int = 3) -> dict:
    """JSON from the Responses API, guarded against truncation, trailing text and missing keys."""
    text_format = {"format": {"type": "json_schema", "name": "out", "schema": schema, "strict": True}}
    for attempt in range(1, attempts + 1):
        status, reply, _ = resilient_call(f"{PREFIX}/responses", {
            "model": model, "input": prompt, "max_output_tokens": budget, "store": False, "text": text_format})
        if status != 200:
            raise RuntimeError(f"HTTP {status}: {err(reply)}")
        if reply.get("status") == "incomplete":
            print(f"  attempt {attempt}: cut off at max_output_tokens={budget}, so the budget goes up")
            budget *= 4
            continue
        parsed = parse_json_lenient(response_text(reply))
        missing = [key for key in schema["required"] if key not in parsed]
        if not missing:
            return parsed
        print(f"  attempt {attempt}: missing {missing}")
    raise RuntimeError(f"no complete JSON after {attempts} attempts")

In [10]:
invoice = structured(MODEL, PROMPT, SCHEMA)
print(json.dumps(invoice))
wrong = {key: invoice.get(key) for key, value in EXPECTED.items() if invoice.get(key) != value}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

print()
for text in ('{"total": 1280.5}', '{"total": 1280.5}\n}', '{"total": 1280.5} and some text after it'):
    try:
        json.loads(text)
        strict = "parses"
    except json.JSONDecodeError:
        strict = "fails"
    print(f"json.loads {strict:6} | parse_json_lenient -> {parse_json_lenient(text)} | {text!r}")

  attempt 1: cut off at max_output_tokens=24, so the budget goes up


{"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}
values match the source

json.loads parses | parse_json_lenient -> {'total': 1280.5} | '{"total": 1280.5}'
json.loads fails  | parse_json_lenient -> {'total': 1280.5} | '{"total": 1280.5}\n}'
json.loads fails  | parse_json_lenient -> {'total': 1280.5} | '{"total": 1280.5} and some text after it'


## 6. A Region check at start-up

Model availability differs by Region, so check at start-up that every model the workload
depends on is in the Region it runs in, and fail before the first request instead of on
it. Presence in the catalogue is not the same as being callable: an entry carries a
`status`, and when that status is not `available` a `status_reason` says why, so read
both rather than the ID alone (section 3). The check below reports an absent model and
an unavailable one separately, because they need different fixes, and the required list
deliberately holds one model this account cannot call, so both outcomes appear below.

A catalogue answers for one endpoint, and that check reads `bedrock-mantle`'s. Its
`GET /v1/models` lists bare model IDs: of the 61 entries in `us-east-1` and the 33 in
`eu-central-1` on 12 November 2026, none began `us.` and none carried a `:` version. A
workload on `bedrock-runtime` sends exactly those two forms, and that endpoint does not
serve this Models API at all, answering 404 `<UnknownOperationException/>`
([`02-migrating-from-openai`](02-migrating-from-openai.ipynb) section 1). So the second
check reads the control plane instead: `ListFoundationModels` for base models, and
`ListInferenceProfiles`, which paginates, for profiles.

The ID `bedrock-runtime` takes for a model depends on the Region, so the check resolves
it per Region with `resolve_runtime_id` rather than hardcoding a geo prefix. On
12 November 2026 `openai.gpt-5.6-sol` resolved to `us.openai.gpt-5.6-sol` in `us-east-1`
and to `global.openai.gpt-5.6-sol` in `eu-central-1`, while
`anthropic.claude-haiku-4-5-20251001-v1:0` resolved to each Region's own profile, `us.`
and `eu.`, and to the `au.` one in `ap-southeast-2`. A geo profile a Region does not
carry is a prefix to change rather than a model that Region lacks, and the last two
checks below keep those apart. Resolution prefers a profile because being listed is not
the same as being callable here either: a base model carries `inferenceTypesSupported`,
and
`anthropic.claude-haiku-4-5-20251001-v1:0` is `ACTIVE` in both Regions with
`['INFERENCE_PROFILE']` alone, so Converse by that bare ID answers
`ValidationException: Invocation of model ID ... with on-demand throughput isn’t
supported.` while `us.anthropic.claude-haiku-4-5-20251001-v1:0` answers 200 (12 November
2026). Check the IDs a workload sends against the catalogue of the endpoint it sends
them to. Those are boto3 calls, so the first cell below sets the client `Config` that
section 2 asks for, and section 7 reuses it.

In [11]:
# claude-fable-5 is listed in us-east-1 with a status of unavailable, so it is here to
# make the check report a model it cannot serve for a reason other than absence.
REQUIRED = [MODEL, "qwen.qwen3-32b", "anthropic.claude-haiku-4-5", "anthropic.claude-fable-5"]


def preflight(required: list[str], region: str) -> list[str]:
    """The required models this Region's bedrock-mantle catalogue cannot serve."""
    status, payload = post("/v1/models", None, region=region, method="GET")
    if status != 200:
        # An unreadable catalogue is not a pass: raise rather than report nothing wrong.
        raise RuntimeError(f"{region}: catalogue unreadable, HTTP {status}: {err(payload)}")
    entries = {entry["id"]: entry for entry in payload.get("data", [])}
    problems = []
    for model in required:
        entry = entries.get(model)
        if entry is None:
            problems.append(f"{model} (absent)")
        elif entry.get("status") != "available":
            reason = entry.get("status_reason") or "no reason given"
            problems.append(f"{model} ({entry.get('status')}: {reason})")
    return problems


for region in ("us-east-1", "eu-central-1"):
    problems = preflight(REQUIRED, region)
    print(f"{region:14} {'PASS' if not problems else 'FAIL'}")
    for problem in problems:
        print(f"  cannot serve on bedrock-mantle: {problem}")

us-east-1      FAIL
  cannot serve on bedrock-mantle: anthropic.claude-fable-5 (unavailable: This model is not available under data retention mode 'default'.)


eu-central-1   FAIL
  cannot serve on bedrock-mantle: anthropic.claude-haiku-4-5 (absent)
  cannot serve on bedrock-mantle: anthropic.claude-fable-5 (absent)


In [12]:
import boto3
from botocore.config import Config

# boto3 1.43.59 defaults to a 60s read timeout, a 60s connect timeout and retry mode
# `legacy`, read off `client.meta.config` on 4 November 2026. 60s is short for a long
# reply, and legacy misses a throttle that arrives as TooManyRequestsException at HTTP
# 400, which `standard` retries by error code (`00-foundations/03`, sections 1 and 2).
BOTO = Config(connect_timeout=10, read_timeout=120,
              retries={"mode": "standard", "max_attempts": 5})

In [13]:
# The models section 7 sends to bedrock-runtime. The ID that endpoint takes for each one
# depends on the Region, so this list holds model IDs and the check resolves them.
RUNTIME_REQUIRED = ["openai.gpt-5.6-sol", "anthropic.claude-haiku-4-5-20251001-v1:0",
                    "amazon.nova-micro-v1:0", "qwen.qwen3-32b-v1:0"]


def preflight_runtime(required: list[str], region: str) -> tuple[dict[str, str], list[str]]:
    """The IDs bedrock-runtime takes here, and the required models it cannot serve."""
    control = boto3.client("bedrock", region_name=region, config=BOTO)
    # Read both catalogues here rather than leaning on the resolver's own lookup: these
    # raise when the control plane refuses, where the resolver falls back to a bare ID.
    base = {m["modelId"]: m for m in control.list_foundation_models()["modelSummaries"]}
    pages = control.get_paginator("list_inference_profiles").paginate()
    profiles = {p["inferenceProfileId"]: p
                for page in pages for p in page["inferenceProfileSummaries"]}
    resolved: dict[str, str] = {}
    problems = []
    for family in required:
        model = resolved[family] = resolve_runtime_id(family, region)
        if model in profiles:
            if profiles[model].get("status") != "ACTIVE":
                problems.append(f"{family} -> {model} (profile {profiles[model].get('status')})")
        elif model in base:
            status = base[model].get("modelLifecycle", {}).get("status")
            types = base[model].get("inferenceTypesSupported", [])
            if status != "ACTIVE":
                problems.append(f"{family} -> {model} ({status})")
            elif "ON_DEMAND" not in types:
                # Reached where a Region lists a model for INFERENCE_PROFILE only and
                # carries no profile for it: the bare ID comes back, and it is refused.
                problems.append(f"{family} -> {model} (listed for {', '.join(types)}, not on demand)")
        else:
            problems.append(f"{family} -> {model} (absent)")
    return resolved, problems


In [14]:
for region in ("us-east-1", "eu-central-1"):
    resolved, problems = preflight_runtime(RUNTIME_REQUIRED, region)
    print(f"{region:14} {'PASS' if not problems else 'FAIL'}")
    for family, model in resolved.items():
        print(f"  {family:42} -> {model}")
    for problem in problems:
        print(f"  cannot serve on bedrock-runtime: {problem}")
# Two failures in eu-central-1 that need different fixes: a geo profile that Region does
# not carry, and a model bedrock-mantle serves and bedrock-runtime does not.
for label, wanted in (("hardcoded geo", "us.openai.gpt-5.6-sol"),
                      ("mantle only", "google.gemma-4-31b")):
    _, problems = preflight_runtime([wanted], "eu-central-1")
    print(f"{label:14} {'PASS' if not problems else 'FAIL'}")
    for problem in problems:
        print(f"  cannot serve on bedrock-runtime: {problem}")


us-east-1      PASS
  openai.gpt-5.6-sol                         -> us.openai.gpt-5.6-sol
  anthropic.claude-haiku-4-5-20251001-v1:0   -> us.anthropic.claude-haiku-4-5-20251001-v1:0
  amazon.nova-micro-v1:0                     -> us.amazon.nova-micro-v1:0
  qwen.qwen3-32b-v1:0                        -> qwen.qwen3-32b-v1:0


eu-central-1   PASS
  openai.gpt-5.6-sol                         -> global.openai.gpt-5.6-sol
  anthropic.claude-haiku-4-5-20251001-v1:0   -> eu.anthropic.claude-haiku-4-5-20251001-v1:0
  amazon.nova-micro-v1:0                     -> eu.amazon.nova-micro-v1:0
  qwen.qwen3-32b-v1:0                        -> qwen.qwen3-32b-v1:0


hardcoded geo  FAIL
  cannot serve on bedrock-runtime: us.openai.gpt-5.6-sol -> us.openai.gpt-5.6-sol (absent)


mantle only    FAIL
  cannot serve on bedrock-runtime: google.gemma-4-31b -> google.gemma-4-31b (absent)


## 7. Guardrails

Amazon Bedrock Guardrails filter requests and replies for denied topics, harmful
content, words and personal data. A guardrail attaches in several ways, and not every
API honours every way: some accept the guardrail header or a body field, return HTTP 200
and apply nothing. So attach the guardrail, then check it on the API you ship. The cells
below create one throwaway guardrail that denies a harmless topic, tulips, so a block
can only come from the guardrail and not from a cautious model; potatoes are the
control. A second signal does not depend on the model: an API that applies guardrails
rejects an identifier that does not exist. `ApplyGuardrail` checks text with no model at
all, so it can screen the request and the reply around an API that ignores the
guardrail, which is what the screening cell below does for `bedrock-mantle`.

Screening both sides is two guardrail calls for one model call, or two per chunk once the
text passes the ceiling below, each billed by text unit of 1,000 characters rounded up: a
200,000-character reply reported 200 `topicPolicyUnits` in
`assessments[0]["invocationMetrics"]["usage"]` in US East (N. Virginia) on 4 November
2026. Each call is also held to a per-policy input size, so the screen below chunks to the
smallest of the ceilings for the policies the guardrail carries (Things that differ). A
chunk boundary can fall mid-sentence, and a policy that reads context can then judge a
chunk differently from the whole text.

The screening cell below sends three prompts. One of them says nothing about tulips and
draws a reply that names them, so the outbound screen is the one that catches it: Gemma 4
answered that way in each of nine runs in US East (N. Virginia) on 31 October 2026, and on
a reply that does not name the topic the cell prints `intervened on: none`. When the guardrail intervenes,
`outputs[0]["text"]` carries its own message, the `blockedOutputsMessaging` on the way
out, and that is the string to return in place of the model's text.

The create call tags the guardrail, because a role can be allowed to create one only when
the request carries a tag. Without `tags`, the same call returned `AccessDeniedException`
naming `bedrock:CreateGuardrail` in US East (N. Virginia) on 12 November 2026, under the
role these notebooks are verified with, and with the tag it returned an identifier. The
name carries a random suffix, and the final cell deletes the guardrail by the identifier
the create call returned, then reads that identifier to confirm it has gone. Deleting what
a name match in `ListGuardrails` returns instead would reach a guardrail another run or
another project created, and that listing returned one guardrail these cells did not
create in the account used here on 12 November 2026. If a run stops between the two cells,
delete the identifier the create cell printed.

In [15]:
from botocore.exceptions import ClientError

control = boto3.client("bedrock", region_name=REGION, config=BOTO)
runtime = boto3.client("bedrock-runtime", region_name=REGION, config=BOTO)
# A suffix per run, so a leftover from an interrupted run does not get in the way and
# this cell has nothing to remove before it creates anything.
GUARDRAIL_NAME = f"bedrock-samples-hardening-demo-{uuid.uuid4().hex[:8]}"
BLOCKED = "BLOCKED_BY_GUARDRAIL"
DENIED = "In two sentences, describe how tulips are cultivated."
ALLOWED = "In two sentences, describe how potatoes are cultivated."

created = control.create_guardrail(
    name=GUARDRAIL_NAME,
    description="Throwaway guardrail for the Amazon Bedrock samples. Safe to delete.",
    topicPolicyConfig={"topicsConfig": [{"name": "Tulips", "type": "DENY", "examples": ["Tell me about tulips."],
                                         "definition": "Any discussion of tulips, tulip bulbs or tulip cultivation."}]},
    blockedInputMessaging=f"{BLOCKED} (input).",
    blockedOutputsMessaging=f"{BLOCKED} (output).",
    # A role can be allowed to create a guardrail only when the request carries a tag,
    # under an aws:RequestTag condition, and untagged this call was then refused.
    tags=[{"key": "created-by", "value": "amazon-bedrock-samples"}],
)
GUARDRAIL = {"guardrailIdentifier": created["guardrailId"], "guardrailVersion": created["version"]}
for _ in range(30):
    state = control.get_guardrail(**GUARDRAIL)["status"]
    if state == "READY":
        break
    time.sleep(2)
else:
    raise RuntimeError(f"the guardrail is not ready: {state}")
print(f"created {GUARDRAIL_NAME} ({created['guardrailId']}), "
      f"version {created['version']}, status {state}")

created bedrock-samples-hardening-demo-46996625 (fz4fmq25sn64), version DRAFT, status READY


In [16]:
for text in (DENIED, ALLOWED):
    result = runtime.apply_guardrail(**GUARDRAIL, source="INPUT", content=[{"text": {"text": text}}])
    topics = [t["name"] for a in result["assessments"] for t in a.get("topicPolicy", {}).get("topics", [])]
    print(f"ApplyGuardrail  {result['action']:22} topics={topics!s:11} {' '.join(text.split()[:6])}")

for model in ("us.amazon.nova-micro-v1:0", "us.anthropic.claude-haiku-4-5-20251001-v1:0"):
    reply = runtime.converse(modelId=model, guardrailConfig=GUARDRAIL, inferenceConfig={"maxTokens": 200},
                             messages=[{"role": "user", "content": [{"text": DENIED}]}])
    text = "".join(b.get("text", "") for b in reply["output"]["message"]["content"])
    print(f"Converse        stopReason={reply['stopReason']:21} blocked={BLOCKED in text}   {model}")
try:
    runtime.converse(modelId="us.amazon.nova-micro-v1:0", inferenceConfig={"maxTokens": 16},
                     guardrailConfig={"guardrailIdentifier": "gr-doesnotexist000", "guardrailVersion": "1"},
                     messages=[{"role": "user", "content": [{"text": "Hi"}]}])
    print("Converse        accepted a guardrail identifier that does not exist")
except ClientError as exc:
    print(f"Converse        an identifier that does not exist -> {exc.response['Error']['Code']}: "
          f"{exc.response['Error']['Message']}")

body = json.dumps({"anthropic_version": "bedrock-2023-05-31", "max_tokens": 200,
                   "messages": [{"role": "user", "content": DENIED}]})
for guarded in (False, True):
    raw = runtime.invoke_model(modelId="us.anthropic.claude-haiku-4-5-20251001-v1:0", body=body,
                               **(GUARDRAIL if guarded else {}))
    reply = json.loads(raw["body"].read())
    text = "".join(b.get("text", "") for b in reply.get("content", []))
    print(f"InvokeModel     guardrail={guarded!s:5} blocked={BLOCKED in text}")

ApplyGuardrail  GUARDRAIL_INTERVENED   topics=['Tulips']  In two sentences, describe how tulips


ApplyGuardrail  NONE                   topics=[]          In two sentences, describe how potatoes


Converse        stopReason=guardrail_intervened  blocked=True   us.amazon.nova-micro-v1:0


Converse        stopReason=guardrail_intervened  blocked=True   us.anthropic.claude-haiku-4-5-20251001-v1:0
Converse        an identifier that does not exist -> ValidationException: The provided guardrail identifier is invalid.


InvokeModel     guardrail=False blocked=False


InvokeModel     guardrail=True  blocked=True


In [17]:
import anthropic
import openai

HOSTS = {"runtime": f"https://bedrock-runtime.{REGION}.amazonaws.com", "mantle": f"https://bedrock-mantle.{REGION}.api.aws"}
SURFACES = {  # name: (endpoint, API, model)
    "runtime Chat Completions": ("runtime", "chat", "qwen.qwen3-32b-v1:0"),
    "runtime Responses": ("runtime", "responses", "us.openai.gpt-5.6-sol"),
    "runtime Messages": ("runtime", "messages", "us.anthropic.claude-haiku-4-5-20251001-v1:0"),
    "mantle Chat Completions": ("mantle", "chat", "qwen.qwen3-32b"),
    "mantle Responses": ("mantle", "responses", "openai.gpt-5.6-sol"),
    "mantle Messages": ("mantle", "messages", "anthropic.claude-haiku-4-5"),
}
HEADER = {"X-Amzn-Bedrock-GuardrailIdentifier": GUARDRAIL["guardrailIdentifier"],
          "X-Amzn-Bedrock-GuardrailVersion": GUARDRAIL["guardrailVersion"]}
BOGUS = {"X-Amzn-Bedrock-GuardrailIdentifier": "gr-doesnotexist000", "X-Amzn-Bedrock-GuardrailVersion": "1"}


def ask(surface: str, headers: dict | None = None, body: dict | None = None) -> tuple[int, str]:
    """Send the tulip question to one API; return (status, reply text). Never raises for a status."""
    endpoint, api, model = SURFACES[surface]
    options = {"extra_headers": headers or {}, "extra_body": body or {}, "timeout": 180}
    question = [{"role": "user", "content": DENIED}]
    key = tokens.get()  # per call, so the provider replaces the key before it expires
    try:
        if api == "messages":
            client = anthropic.Anthropic(base_url=HOSTS[endpoint] + "/anthropic", auth_token=key, api_key=None)
            reply = client.messages.create(model=model, max_tokens=300, messages=question, **options)
            return 200, "".join(b.text for b in reply.content if b.type == "text")
        client = openai.OpenAI(base_url=HOSTS[endpoint] + api_prefix(model, endpoint), api_key=key)
        if api == "responses":
            return 200, client.responses.create(model=model, input=DENIED, max_output_tokens=2000, store=False,
                                                **options).output_text
        reply = client.chat.completions.create(model=model, messages=question, max_completion_tokens=2000, **options)
        return 200, reply.choices[0].message.content or ""
    except (openai.APIStatusError, anthropic.APIStatusError) as exc:
        return exc.status_code, ""
    except (openai.APIConnectionError, anthropic.APIConnectionError):
        return -1, ""

In [18]:
def outcome(status: int, text: str) -> str:
    """blocked or ignored for a 200, refused for a 4xx, and a marked status when the call proves nothing."""
    if status == 200:
        return "blocked" if BLOCKED in text else "ignored"
    return f"refused {status}" if 400 <= status < 500 and status != 429 else f"{status}?"


def measure(surface: str) -> dict:
    """The control, the guardrail header, an unknown guardrail ID, and the body field, on one API."""
    control_status, control_text = ask(surface)
    bogus = outcome(*ask(surface, headers=BOGUS))
    return {"surface": surface,
            "control": "answered" if control_status == 200 and len(control_text) > 40 and BLOCKED not in control_text
            else outcome(control_status, control_text),
            "header": outcome(*ask(surface, headers=HEADER)),
            "unknown id": "rejected" if bogus.startswith("refused") else "accepted" if bogus == "ignored" else bogus,
            "body field": outcome(*ask(surface, body={"guardrailConfig": GUARDRAIL}))}


with cf.ThreadPoolExecutor(max_workers=6) as pool:
    rows = list(pool.map(measure, SURFACES))
COLUMNS = ("control", "header", "unknown id", "body field")
print(f"{'API':26}" + "".join(f"{column:>14}" for column in COLUMNS))
for row in rows:
    print(f"{row['surface']:26}" + "".join(f"{row[column]:>14}" for column in COLUMNS))

print("\ncomputed from the table:")
for label, test in (("the header is applied on", lambda r: r["header"] == "blocked"),
                    ("the header is accepted and ignored on", lambda r: r["header"] == "ignored"),
                    ("the body field is ignored on", lambda r: r["body field"] == "ignored"),
                    ("the body field is refused on", lambda r: r["body field"].startswith("refused"))):
    print(f"  {label:38} {', '.join(r['surface'] for r in rows if test(r)) or 'none'}")
disagree = [r["surface"] for r in rows if (r["header"] == "blocked") != (r["unknown id"] == "rejected")]
print(f"  {'the two signals disagree on':38} {', '.join(disagree) or 'none'}")
unanswered = [r["surface"] for r in rows if r["control"] != "answered"]
if unanswered:
    print("  !! no usable control answer, so these rows prove nothing:", unanswered)

API                              control        header    unknown id    body field
runtime Chat Completions        answered       blocked      rejected       ignored
runtime Responses               answered       blocked      rejected   refused 400
runtime Messages                answered       blocked      rejected   refused 400
mantle Chat Completions         answered       ignored      accepted       ignored
mantle Responses                answered       ignored      accepted   refused 400
mantle Messages                 answered       ignored      accepted   refused 400

computed from the table:
  the header is applied on               runtime Chat Completions, runtime Responses, runtime Messages
  the header is accepted and ignored on  mantle Chat Completions, mantle Responses, mantle Messages
  the body field is ignored on           runtime Chat Completions, mantle Chat Completions
  the body field is refused on           runtime Responses, runtime Messages, mantle Responses, man

In [19]:
# bedrock-mantle takes the guardrail header and applies nothing, as the table above shows,
# and it does not serve Converse, so the client screens both sides itself. The second prompt
# is why the outbound screen is not optional: it says nothing about the denied topic and the
# reply names it, so the inbound screen has nothing to catch.
DRIFTS = "Name three well-known Dutch exports, as a comma-separated list."

# One ApplyGuardrail request is held to a per-policy input size, a unit being 1,000
# characters rounded up. This guardrail is on the Classic tier, because the create call
# above sets no topicPolicyConfig["tierConfig"], and a Classic denied topic policy allows
# 200 text units: 200,000 characters pass, 200,001 fail. The same policy on the Standard
# tier allows 1,000, and Standard needs cross-Region inference, so moving to it means
# passing crossRegionConfig as well (us-east-1, 4 November 2026). A request is charged
# against every policy the guardrail carries and fails at the lowest of their ceilings, so
# chunk to the smallest quota among the policies you configure rather than the one you
# happen to look up: contextual grounding is the smallest at 106 text units and is not
# tiered, while content, word and sensitive information are 1,000. Each is an adjustable
# Service Quota, so read yours (Things that differ) rather than trusting this number.
SCREEN_LIMIT = 200_000  # characters; this guardrail carries one Classic denied topic policy


def screen(text: str, source: str) -> str | None:
    """The guardrail's replacement for this text, or None when it does not intervene.

    Screens in SCREEN_LIMIT chunks, and returns on the first chunk that intervenes, which
    is the whole reply for a policy that blocks. A policy that masks needs every chunk
    screened and the masked chunks joined, or the text after the first masked chunk is
    lost.
    """
    # max(..., 1) so empty text is screened once rather than skipped by an empty range.
    for start in range(0, max(len(text), 1), SCREEN_LIMIT):
        chunk = text[start:start + SCREEN_LIMIT]
        checked = runtime.apply_guardrail(**GUARDRAIL, source=source,
                                          content=[{"text": {"text": chunk}}])
        if checked["action"] == "GUARDRAIL_INTERVENED":
            # An intervention carries its own text: the canned message where the policy
            # blocks, the masked text where it masks. Either way that is what the caller
            # should receive.
            return (checked.get("outputs") or [{"text": BLOCKED}])[0]["text"]
    return None


def screened(prompt: str) -> tuple[str, str, str]:
    """One mantle Responses call with ApplyGuardrail on the way in and on the way out."""
    stopped = screen(prompt, "INPUT")
    if stopped is not None:
        return stopped, "input", "not called"
    status, reply, _ = resilient_call(f"{PREFIX}/responses",
                                      {"model": MODEL, "input": prompt, "max_output_tokens": 300, "store": False})
    if status != 200:
        # With no answer there is nothing to screen, so do not report a pass.
        raise RuntimeError(f"the model call failed, so the outbound screen proves nothing: {status} {err(reply)}")
    answer = response_text(reply).strip()
    stopped = screen(answer, "OUTPUT")
    return answer if stopped is None else stopped, "none" if stopped is None else "output", reply.get("status", "?")


for prompt in (DENIED, DRIFTS, ALLOWED):
    text, where, reply_status = screened(prompt)
    print(f"{' '.join(prompt.split()[:6])}  intervened on: {where}, reply status: {reply_status}")
    print(f"  the caller receives: {text}")


In two sentences, describe how tulips  intervened on: input, reply status: not called
  the caller receives: BLOCKED_BY_GUARDRAIL (input).


Name three well-known Dutch exports, as  intervened on: output, reply status: completed
  the caller receives: BLOCKED_BY_GUARDRAIL (output).


In two sentences, describe how potatoes  intervened on: none, reply status: completed
  the caller receives: Potatoes are cultivated by planting seed tubers or cuttings in nutrient-rich, well-draining soil. They are then grown under consistent moisture and sunlight until the foliage dies back, signaling that the tubers are ready to be harvested from the earth.


In [20]:
# Delete by the identifier the create call returned, and confirm by reading that
# identifier. Deleting what a name match in ListGuardrails returns would reach a guardrail
# another run or another project created: that listing answers for the whole account.
control.delete_guardrail(guardrailIdentifier=GUARDRAIL["guardrailIdentifier"])
for _ in range(15):
    try:
        state = control.get_guardrail(**GUARDRAIL)["status"]
    except ClientError as exc:
        if exc.response["Error"]["Code"] != "ResourceNotFoundException":
            raise
        print(f"deleted {GUARDRAIL['guardrailIdentifier']}: GetGuardrail now returns "
              f"{exc.response['Error']['Code']}")
        break
    time.sleep(2)
else:
    raise RuntimeError(f"{GUARDRAIL['guardrailIdentifier']} is still {state}; "
                       "delete it in the console")


deleted fz4fmq25sn64: GetGuardrail now returns ResourceNotFoundException


## 8. A pre-launch checklist

1. Keys are minted in the process from the role, with a short lifetime, and no long-term
   API key is deployed.
2. 429, 5xx and 529 are retried with backoff and jitter; other 4xx fail at once.
3. Every call has a client timeout, and retries are bounded, the boto3 clients
   included: both come from a `botocore.config.Config` on the client, not from the call.
4. Load rises over minutes, not in one step ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)).
5. `store` is set explicitly on every Responses API request.
6. The data retention mode is chosen, and every model in use allows it.
7. Each workload runs under its own tagged Project ([`00-foundations/02`](../00-foundations/02-governance-projects-and-retention.ipynb)).
8. Dashboards for `bedrock-mantle` traffic use `AWS/BedrockMantle`, and the client counts
   5xx, 429s and latency itself, because that namespace does not publish them.
9. CloudTrail data events are on if you need a record of inference calls.
10. Output is checked for truncation, parsed leniently, and its values are checked.
11. Start-up fails when a required model is missing from the Region, checked on the
    endpoint the workload calls: `bedrock-mantle`'s catalogue for the bare IDs it takes,
    with a `status` of `available`, and `ListFoundationModels` plus
    `ListInferenceProfiles` for the profiles and versioned IDs `bedrock-runtime` takes,
    with an `ACTIVE` status and, for a base model, `ON_DEMAND` among its
    `inferenceTypesSupported`. The runtime ID is resolved per Region rather than
    hardcoded, so a Region whose profiles carry another geo prefix still starts.
12. Guardrails are verified on the exact API in production, and where that API accepts
    them without applying them, the client screens the request and the reply with
    `ApplyGuardrail`, chunked to the smallest per-policy input ceiling the guardrail
    carries.
13. Sampling parameters and service tiers are sent only where each model accepts them.
14. A throwaway resource a run creates carries a name of its own and is deleted by the
    identifier the create call returned, so the cleanup deletes that resource and nothing
    else.

## Things that differ

What trips people up here, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| An expired key | Calls fail once a short-term key passes its lifetime, at most 12 hours | Mint a new key before expiry (section 1). |
| Throttling and capacity | 429, 5xx, and 529 from the Messages API | Retry with backoff and jitter (section 2). |
| A server fault with a client status | 400 with `Internal server error` in the body | Retry it; fail fast on every other 4xx. |
| A path that stalls | No reply at all, for example Grok 4.3 on `/v1/responses` | Set a client timeout and bound the retries. |
| boto3's client defaults | A bare `boto3.client("bedrock-runtime")` reports `read_timeout` 60, `connect_timeout` 60 and `retries {'mode': 'legacy'}` (boto3 1.43.59, 4 November 2026) | Pass `Config(connect_timeout=..., read_timeout=..., retries={"mode": "standard", "max_attempts": ...})` to every Bedrock client. 60s is short for a long reply, and legacy leaves a throttle delivered as `TooManyRequestsException` at HTTP 400 unretried ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| A retention mode a model does not allow | `status` `unavailable`: `This model is not available under data retention mode 'default'.` | Read `status` and `allowed_modes` before debugging the request (section 3). |
| Server errors in CloudWatch | No 5xx metric and no latency metric in `AWS/BedrockMantle` | Count them in the client (section 4). |
| Throttles in CloudWatch | No throttle metric in `AWS/BedrockMantle`, and `InferenceClientErrors` leaves out requests rejected before processing, quota-related 429s included, so throttling does not show there either (`us-east-1`, 19 October 2026) | Count 429s in the client, as the recorder in section 4 does. |
| A Region check that reads the wrong catalogue | `bedrock-mantle`'s `GET /v1/models` lists bare IDs only: among its 61 `us-east-1` entries and 33 `eu-central-1` entries none began `us.` and none carried a `:` version, while `bedrock-runtime` lists `anthropic.claude-haiku-4-5-20251001-v1:0` `ACTIVE` for `INFERENCE_PROFILE` alone and Converse by that ID answers `ValidationException: Invocation of model ID anthropic.claude-haiku-4-5-20251001-v1:0 with on-demand throughput isn’t supported. Retry your request with the ID or ARN of an inference profile that contains this model.` (12 November 2026) | Check each endpoint's own catalogue for the IDs you send it (section 6): `GET /v1/models` for `bedrock-mantle`, and `ListFoundationModels` plus `ListInferenceProfiles` for `bedrock-runtime`, reading `inferenceTypesSupported` on a base model. Resolve the runtime ID per Region rather than hardcoding a geo prefix: `openai.gpt-5.6-sol` is `us.openai.gpt-5.6-sol` in `us-east-1` and `global.openai.gpt-5.6-sol` in `eu-central-1` (12 November 2026), so a `us.` profile missing from a Region is a prefix to change rather than a model that Region lacks. |
| A 200 that is not a result | A reply cut off at the limit, text after the JSON, or a wrong value | Check `status`, parse leniently and check the values (section 5). |
| Guardrails that are accepted and not applied | HTTP 200 and no block | Verify on the API you ship, and where it does not apply them screen the request and the reply yourself with `ApplyGuardrail` (section 7). Converse does apply them, and `bedrock-mantle` does not serve Converse. |
| A cleanup that deletes what a list returns | `ListGuardrails` answers for the whole account, and returned one guardrail these cells did not create in the account used here (`us-east-1`, 12 November 2026) | Give a throwaway resource a name of its own per run, delete it by the identifier its create call returned, and confirm by reading that identifier: `GetGuardrail` answered `DELETING` and then `ResourceNotFoundException: This guardrail does not exist.` about two seconds later (`us-east-1`, 12 November 2026). Section 7 does this. |
| A create refused for the want of a tag | `AccessDeniedException: User: arn:aws:sts::123456789012:assumed-role/<role>/<session> is not authorized to perform: bedrock:CreateGuardrail` (account ID, role and session replaced) on a call with no `tags`, where the same call with `tags=[{"key": "created-by", "value": "amazon-bedrock-samples"}]` returned a guardrail identifier (`us-east-1`, 12 November 2026) | Read the conditions on your own role before reading this as a missing permission. `bedrock:CreateGuardrail` can be granted under an `aws:RequestTag` condition and `DeleteGuardrail` under the matching `aws:ResourceTag`, and an untagged request is then refused with the message a missing permission gives. |
| An `ApplyGuardrail` request over the input ceiling | `ThrottlingException: Input text size (210 text units) exceeds the maximum allowed (200 text units) for the denied topic policy (Classic tier).` at 210,000 characters, where 200,000 passes and reports `topicPolicyUnits` 200 (`us-east-1`, 4 November 2026) | Chunk the text before screening, as section 7 does, and chunk to the lowest ceiling among the policies the guardrail carries: every one of them is charged the whole input, so a guardrail with a denied topic and a content filter fails at the topic policy's 200 while the content policy would have allowed 1,000. Do not retry this one: it carries a throttle's error code at HTTP 429 and the cause is the size, so a retry policy spends every attempt on it, 5 attempts under boto3's default `legacy` mode and 6 under the `standard` mode that section 7's client sets, because botocore reads `max_attempts=5` as a retry count and reports `total_max_attempts` 6 (5.8s and 19.3s on a 400,000-character input, `us-east-1`, 4 November 2026). The ceiling is an adjustable Service Quota, one per policy and per tier: `(Guardrails) Topic policy maximum input size in text units (Classic tier)` is 200 and the `(Standard tier - Recommended)` one is 1,000, so read the quota for the tier your guardrail is on. |

## Next

- Which model and API, measured across families:
  [`01-choosing-a-model-and-api`](01-choosing-a-model-and-api.ipynb)
- Moving existing OpenAI or Anthropic code:
  [`02-migrating-from-openai`](02-migrating-from-openai.ipynb)
- Quotas, retries and service tiers:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)